# RAG Token-Budget — Colab runner
**QM640 capstone experiment.** This notebook runs the paid pipeline stages on Colab
(the Mac freezes on the spaCy/LLMLingua compute) and syncs every artifact back to
Google Drive after each stage, so an interrupted session resumes for free.

## One-time setup
1. Upload `colab_bundle.zip` (created on the Mac, in the repo root) to your Google Drive at
   **`MyDrive/ragtb/colab_bundle.zip`**.
2. Add your OpenRouter key in Colab: click the key icon (left sidebar) → **Secrets** →
   add `OPENROUTER_API_KEY` with notebook access enabled.
3. Runtime → Change runtime type → **T4 GPU** (speeds up the LLMLingua arm a lot; CPU works too).

## If the session disconnects
Just run all cells again top to bottom. Everything (embeddings, generations, judgements,
assembled contexts, block checkpoints) is cached in `llm_cache/cache.db` and
`data/eval_partial/`, both synced to Drive — nothing already paid is ever re-paid.

## Getting results back to the Mac
After the final cell, `MyDrive/ragtb/sync/` holds `outputs/`, `data/` results, the cache DB
and `EXPERIMENT_LOG.md`. Download that folder into the Mac repo (details in the last cell).


In [ ]:
#@title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/ragtb')
SYNC = DRIVE / 'sync'
SYNC.mkdir(parents=True, exist_ok=True)
assert (DRIVE / 'colab_bundle.zip').exists(), (
    'Upload colab_bundle.zip to MyDrive/ragtb/ first (see setup instructions above)')
print('Drive OK:', DRIVE)

In [ ]:
#@title 2. Unpack the project (+ restore any previous Colab progress)
import shutil, subprocess, os, pathlib

ROOT = pathlib.Path('/content/rag-token-budget')
if not ROOT.exists():
    subprocess.run(['unzip', '-q', str(DRIVE / 'colab_bundle.zip'), '-d', '/content'], check=True)
assert ROOT.exists(), 'unzip did not produce /content/rag-token-budget'

# restore progress from a previous Colab session (cache DB, checkpoints, outputs)
def restore(rel):
    src = SYNC / rel
    dst = ROOT / rel
    if src.is_file() and (not dst.exists() or src.stat().st_size > dst.stat().st_size):
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print('restored', rel)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.rglob('*'):
            if f.is_file():
                t = dst / f.relative_to(src)
                t.parent.mkdir(parents=True, exist_ok=True)
                if not t.exists() or f.stat().st_mtime > t.stat().st_mtime:
                    shutil.copy2(f, t)
        print('restored dir', rel)

for rel in ['llm_cache/cache.db', 'data/eval_partial', 'data/eval_records.parquet',
            'data/baseline_records.parquet', 'data/position_records.parquet',
            'data/embeddings.npy', 'data/chunk_ids.json', 'data/question_embeddings.npy',
            'data/question_ids.json', 'data/bm25.pkl', 'data/manifest.json',
            'data/graph_edges.parquet', 'data/chunk_entities.parquet',
            'outputs', 'EXPERIMENT_LOG.md']:
    restore(rel)

os.chdir(ROOT)
print('project at', ROOT)

In [ ]:
#@title 3. Install dependencies (~2-4 min first time)
%pip -q install -r requirements.txt
import spacy, llmlingua, tabulate, torch
print('GPU available:', torch.cuda.is_available())
spacy.load('en_core_web_sm')
print('deps OK')

In [ ]:
#@title 4. Configure environment (.env from Colab secret; no Postgres on Colab)
from google.colab import userdata
import os, pathlib

key = userdata.get('OPENROUTER_API_KEY')
assert key and key.startswith('sk-or-'), 'Add OPENROUTER_API_KEY to Colab Secrets (key icon, left sidebar)'
pathlib.Path('.env').write_text(f'OPENROUTER_API_KEY={key}\n')

os.environ['PYTHONHASHSEED'] = '0'      # reproducibility, matches run_all.sh
os.environ['SKIP_PGVECTOR'] = '1'       # no Postgres server on Colab — recorded
                                        # deviation; retrieval is exact in-memory
                                        # cosine either way (see manifest)
ENV = dict(os.environ)
print('env ready (key set, SKIP_PGVECTOR=1, PYTHONHASHSEED=0)')

In [ ]:
#@title 5. Define run + sync helpers
import subprocess, shutil, sys, time, pathlib

def run_stage(cmd):
    """Run a pipeline stage, streaming output into the notebook; fail loudly."""
    print('$', ' '.join(cmd), flush=True)
    t0 = time.time()
    p = subprocess.run([sys.executable] + cmd, env=ENV)
    print(f'--- exit {p.returncode} in {(time.time()-t0)/60:.1f} min', flush=True)
    if p.returncode != 0:
        raise RuntimeError(f'{cmd[0]} failed (exit {p.returncode}) — see output above')

def sync_to_drive():
    """Copy every result artifact to Drive so a disconnect loses nothing."""
    for rel in ['llm_cache/cache.db', 'data/eval_partial', 'data/eval_records.parquet',
                'data/baseline_records.parquet', 'data/position_records.parquet',
                'data/embeddings.npy', 'data/chunk_ids.json', 'data/question_embeddings.npy',
                'data/question_ids.json', 'data/bm25.pkl', 'data/manifest.json',
                'data/graph_edges.parquet', 'data/chunk_entities.parquet',
                'data/passages_clean.parquet', 'data/sample',
                'outputs', 'EXPERIMENT_LOG.md']:
        src = pathlib.Path(rel)
        dst = SYNC / rel
        if src.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
        elif src.is_dir():
            for f in src.rglob('*'):
                if f.is_file():
                    t = dst / f.relative_to(src)
                    t.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(f, t)
    print('synced to', SYNC, flush=True)

print('helpers ready')

In [ ]:
#@title 6. Stages 01+02 — data + cleaning (skip instantly: artifacts shipped in the bundle)
run_stage(['src/01_acquire.py'])
run_stage(['src/02_clean.py'])

In [ ]:
#@title 7. Stage 03 — embeddings + retrieval gate (re-derives from cache, ~$0 if already run on the Mac)
run_stage(['src/03_index.py', '--yes'])
sync_to_drive()

In [ ]:
#@title 8. Stage 04 — chunk graph (spaCy NER, the step that froze the Mac) + budget compliance (~$0.25)
run_stage(['src/04_arms.py', '--yes'])
sync_to_drive()

In [ ]:
#@title 9. Stage 05 SMOKE TEST — 20 questions, all arms & budgets (~$1)
run_stage(['src/05_run.py', '--limit', '20', '--yes'])
sync_to_drive()

import pandas as pd
ev = pd.read_parquet('data/eval_records.parquet')
print(f'{len(ev)} smoke records')
display(ev.pivot_table(index='arm', columns='budget', values='em', aggfunc='mean').round(3))
display(ev[['arm','budget','predicted_answer','gold_answer','em','f1','faithfulness']]
        .sample(min(12, len(ev)), random_state=0))

### ⏸️ Inspect the smoke test above before continuing
Sanity checks: answers look like real short answers (not refusals/empty), EM/F1 nonzero,
faithfulness mostly parsed. If something looks broken, stop here and investigate —
the cells below spend the real budget (~$23–26 ceiling, cache-discounted on re-runs).

In [ ]:
#@title 10. Stage 05 FULL SWEEPS — 6 arms x 4 budgets x (600 primary + 600 structured) + judge + hops-1 sensitivity (~3-5 h, checkpoint-synced every block)
import threading, time as _t

# background sync every 10 min so long sweeps survive disconnects mid-stage
_stop = threading.Event()
def _auto():
    while not _stop.is_set():
        _stop.wait(600)
        try: sync_to_drive()
        except Exception as e: print('sync skipped:', e)
t = threading.Thread(target=_auto, daemon=True); t.start()

try:
    run_stage(['src/05_run.py', '--yes', '--workers', '12'])
finally:
    _stop.set(); t.join(timeout=5)
    sync_to_drive()

In [ ]:
#@title 11. Stage 04b — reference conditions (floor/ceiling/random/full-context)
run_stage(['src/04b_baselines.py', '--yes', '--workers', '12'])
sync_to_drive()

In [ ]:
#@title 12. Stage 04c — lost-in-the-middle position ablation
run_stage(['src/04c_position.py', '--yes', '--workers', '12'])
sync_to_drive()

In [ ]:
#@title 13. Stages 06 + 07 + 08 — analysis, power, RESULTS_SUMMARY (free, no API)
run_stage(['src/06_analyze.py'])
run_stage(['src/07_power.py'])
run_stage(['src/08_summary.py'])
sync_to_drive()

In [ ]:
#@title 14. Results — figures inline + summary
from IPython.display import Image, Markdown, display
import pathlib

for fig in ['fig_pareto.png', 'fig_accuracy_by_budget.png', 'fig_hop_breakdown.png',
            'fig_by_dataset.png', 'fig_structured_vs_prose.png', 'fig_latency_cost.png',
            'fig_position_effect.png', 'fig_power_curve.png']:
    p = pathlib.Path('outputs') / fig
    if p.exists():
        print(fig)
        display(Image(str(p), width=900))

display(Markdown(pathlib.Path('outputs/RESULTS_SUMMARY.md').read_text()))

## Bringing everything back to the Mac repo
All artifacts are in **`MyDrive/ragtb/sync/`**. On the Mac:

```bash
# with Google Drive for desktop (or download the sync/ folder manually):
SYNC=~/Library/CloudStorage/GoogleDrive-*/My\ Drive/ragtb/sync
cd /Volumes/Esmaeil/PROJECTS/UNI/rag-token-budget
cp -R "$SYNC/outputs/." outputs/
cp -R "$SYNC/data/." data/
cp "$SYNC/llm_cache/cache.db" llm_cache/cache.db
cp "$SYNC/EXPERIMENT_LOG.md" EXPERIMENT_LOG.md
```

Then tell Claude Code the results are back — it will verify the artifacts, update the
experiment log, and commit `outputs/` (the report cites those files).